<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/ml/notebooks/c5_l2.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C5-L2 · Etiquetado y embargo
Etiqueta de horizonte fijo + triple barrera; el embargo desinfla el score tramposo.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/ml/data/c5_l2.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c5_l2.csv'), Path('data/c5_l2.csv'), Path('c5_l2.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
# Etiqueta horizonte fijo h=3 menos costos; triple barrera simple (TP/SL/tmax)
H, COSTO, TP, SL = 3, 0.001, 0.02, 0.015
rets = df['close'].pct_change()
df['ret_fut'] = sum(df['close'].shift(-k)/df['close'].shift(-k+1) - 1 for k in range(1, H+1))
df['y_fijo'] = (df['ret_fut'] - COSTO > 0).astype(int)
# Triple barrera: recorre H dias, gana TP, pierde SL, si no, signo al vencimiento
outs = []
for i in range(len(df)):
    r = 0.0; out = 0
    for k in range(1, H+1):
        if i+k >= len(df): break
        r = df['close'].iloc[i+k]/df['close'].iloc[i] - 1
        if r >= TP: out = 1; break
        if r <= -SL: out = 0; break
    else:
        out = int(r - COSTO > 0)
    outs.append(out)
df['y_tb'] = outs
print(df[['dia','close','ret_fut','y_fijo','y_tb']].head(8).to_string(index=False))
print('positivos fijo:', int(df["y_fijo"].sum()), ' triple-barrera:', int(df["y_tb"].sum()))
assert set(df['y_fijo'].dropna().unique()) <= {0, 1}
assert set(df['y_tb'].unique()) <= {0, 1}

In [ ]:
# Features honestas (lags) para predecir la etiqueta
for k in range(1, 6):
    df[f'lag_{k}'] = df['close'].shift(k)
feat = [f'lag_{k}' for k in range(1, 6)]
data = df.dropna().reset_index(drop=True)
from sklearn.linear_model import LogisticRegression
X = data[feat].values; y = data['y_tb'].values
split = int(len(data)*0.7)
# Modelo ingenuo SIN embargo vs modelo CON embargo (purga 3 filas del train)
from sklearn.metrics import accuracy_score
base = LogisticRegression(max_iter=500).fit(X[:split], y[:split])
acc_sin = accuracy_score(y[split:], base.predict(X[split:]))
EMB = 3
emb = LogisticRegression(max_iter=500).fit(X[:split-EMB], y[:split-EMB])
acc_emb = accuracy_score(y[split:], emb.predict(X[split:]))
print(f'acc sin embargo={acc_sin:.3f}  acc con embargo={acc_emb:.3f}')
assert 0.0 <= acc_emb <= 1.0 and 0.0 <= acc_sin <= 1.0
print('OK: ambas variantes evaluadas')

In [ ]:
# El embargo no debe inflar: con purga el score no supera al ingenuo + margen
print(f'diferencia (sin-embargo menos con-embargo) = {acc_sin-acc_emb:+.3f}')
assert acc_emb <= acc_sin + 0.15, 'el embargo no debe inflar el score'

In [ ]:
# Chequeo automatico L2
assert H == 3 and len(data) > 30
assert abs(df['ret_fut'].iloc[0] - ((df['close'].iloc[1]/df['close'].iloc[0]-1)+(df['close'].iloc[2]/df['close'].iloc[1]-1)+(df['close'].iloc[3]/df['close'].iloc[2]-1))) < 1e-9
assert set(data['y_tb'].unique()) <= {0, 1}
print('OK L2: etiquetado + embargo verificados')